# Lecture 1 — SOLUTIONS: EDA and basic programming by decomposition

**Guiding idea.** Before writing any code, break the problem down into smaller and smaller sub-problems,
until every leaf of the tree is simple enough to become **a function** (or a few lines of code).
In project management this is called a *Work Breakdown Structure*; in computer science we talk about
**functional decomposition**, *top-down design* or *stepwise refinement* (Wirth, 1971).

For each exercise:
1. read the question;
2. complete/discuss the **decomposition tree** (in markdown, before the code);
3. write **one function per leaf**, then a "root" function that calls them in order.

**Datasets** (in the `Datasets/` folder at the root of the repository):
- `penguins.csv` — Palmer Penguins: 344 penguins, 3 species, bill, flipper and body mass measurements (Antarctica, 2007–2009).
- `titanic.csv` — 891 Titanic passengers with class, sex, age, fare and survival.
- `icecream_sharks.csv`, `berkeley_admissions.csv`, `galton_heights.csv`, `anscombe.csv` — four small datasets for Exercise 6 on statistical traps.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

penguins = pd.read_csv("../Datasets/penguins.csv")
titanic = pd.read_csv("../Datasets/titanic.csv")

## Exercise 1 — Is the dataset ready for analysis? (penguins)

**Question:** can I trust `penguins.csv` as it is?

```
Is the dataset ready?
├── How big is it?              → number of rows and columns
├── What type is each column?   → numerical vs categorical
├── Are there missing values?   → count and % per column
└── Are there duplicates?       → number of duplicated rows
```

**To do:** write `quality_report(df)` that returns a DataFrame with, for each column:
`type`, `n_missing`, `pct_missing`, `n_unique`; and prints the shape and the number of duplicates.

*Discussion question:* should the rows with a missing `sex` be dropped? What about those with all measurements missing?

In [ ]:
def quality_report(df):
    print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
    print(f"Duplicated rows: {df.duplicated().sum()}")
    report = pd.DataFrame({
        "type": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "pct_missing": (df.isna().mean() * 100).round(1),
        "n_unique": df.nunique(),
    })
    return report

quality_report(penguins)

## Exercise 2 — Descriptive statistics by group (penguins)

**Question:** how much do penguins of the different species weigh, and is there a difference between males and females?

```
Body mass comparison
├── Clean       → drop rows without body mass or without sex
├── Group       → by species (and then by species + sex)
├── Summarize   → n, mean, median, standard deviation
└── Present     → sorted and rounded table
```

**To do:** write `summary_by_group(df, groups, column)`, reusable with any numerical column
(try it with `body_mass_g` and then with `flipper_length_mm`).

*Discussion question:* are mean and median similar? What does this tell you about the shape of the distribution?

In [ ]:
def clean(df, columns):
    return df.dropna(subset=columns)

def summary_by_group(df, groups, column):
    data = clean(df, groups + [column])
    table = (data.groupby(groups)[column]
                 .agg(["count", "mean", "median", "std"])
                 .round(1)
                 .sort_values("mean", ascending=False))
    return table

display(summary_by_group(penguins, ["species"], "body_mass_g"))
display(summary_by_group(penguins, ["species", "sex"], "body_mass_g"))

## Exercise 3 — Look before you conclude (penguins)

**Question:** are bill length and bill depth correlated? In which direction?

```
Relationship between two measurements
├── Overall correlation         → one number for the whole dataset
├── Correlation by species      → one number per species
└── Plot                        → scatter plot coloured by species
```

**To do:** write `correlation(df, x, y, group=None)` and `scatter_by_group(df, x, y, group)`,
then compare the results with `x="bill_length_mm"`, `y="bill_depth_mm"`, `group="species"`.

*Discussion question:* is the sign of the overall correlation the same as the one within each species?
What is this phenomenon called? (hint: it starts with "S")

In [ ]:
def correlation(df, x, y, group=None):
    data = df.dropna(subset=[x, y])
    if group is None:
        return round(data[x].corr(data[y]), 2)
    return data.groupby(group).apply(lambda g: round(g[x].corr(g[y]), 2))

def scatter_by_group(df, x, y, group):
    fig, ax = plt.subplots(figsize=(6, 4.5))
    for name, g in df.groupby(group):
        ax.scatter(g[x], g[y], label=name, alpha=0.7, s=18)
    ax.set_xlabel(x); ax.set_ylabel(y); ax.legend(title=group)
    plt.show()

print("Overall:", correlation(penguins, "bill_length_mm", "bill_depth_mm"))
print(correlation(penguins, "bill_length_mm", "bill_depth_mm", "species"))
scatter_by_group(penguins, "bill_length_mm", "bill_depth_mm", "species")
# Overall ≈ -0.24, within each species ≈ +0.4/+0.65: Simpson's paradox.

## Exercise 4 — Who survived? (titanic)

**Question:** did the probability of surviving depend on sex and class?

```
Survival rate
├── By sex                     → % survivors
├── By class                   → % survivors
├── By sex × class             → two-way table
└── Plot                       → grouped bar chart
```

**To do:** write `survival_rate(df, groups)` that returns the percentage of survivors
(the `survived` column is 0/1: what is the trick to get the percentage?) and a function that draws
the bar chart of the sex × class table.

*Discussion question:* the dataset has redundant columns (`survived`/`alive`, `pclass`/`class`,
`embarked`/`embark_town`). Why is this a problem, and which ones do you keep?

In [ ]:
def survival_rate(df, groups):
    # the mean of a 0/1 column is the proportion of 1s
    return (df.groupby(groups)["survived"].mean() * 100).round(1)

def survival_bars(df, rows, columns):
    table = survival_rate(df, [rows, columns]).unstack()
    ax = table.plot(kind="bar", figsize=(6, 4), rot=0)
    ax.set_ylabel("% survivors"); ax.set_ylim(0, 100)
    plt.show()
    return table

display(survival_rate(titanic, ["sex"]))
display(survival_rate(titanic, ["class"]))
survival_bars(titanic, "class", "sex")

## Exercise 5 — A small complete pipeline (titanic)

**Question:** were children more likely to survive than adults? And did travelling alone help?

This time **you draw the tree** (in a markdown cell) before writing any code. Constraints:
- age is missing for about 20% of the passengers: decide what to do and write it in the tree;
- create the columns `age_group` (e.g. 0–12, 13–17, 18–39, 40–59, 60+) and `family_size` (`sibsp + parch + 1`);
- the root function `analysis(df)` must call, in order: `clean` → `add_features` → `summarize`
  and contain no other logic.

*Discussion question:* if you fill the missing ages with the median, what happens to the age group
the median falls into? Does the conclusion change?

In [ ]:
# Tree:
# analysis
# ├── clean          → keep only the useful columns, drop rows without age (declared choice)
# ├── add_features   → age_group (pd.cut), family_size, travels_alone
# └── summarize      → survival rate by age_group and by family_size

def clean(df):
    columns = ["survived", "sex", "pclass", "age", "sibsp", "parch", "fare"]
    return df[columns].dropna(subset=["age"]).copy()

def add_features(df):
    df = df.copy()
    df["age_group"] = pd.cut(df["age"], bins=[0, 12, 17, 39, 59, 120],
                             labels=["0-12", "13-17", "18-39", "40-59", "60+"])
    df["family_size"] = df["sibsp"] + df["parch"] + 1
    df["travels_alone"] = df["family_size"] == 1
    return df

def summarize(df):
    by_age = df.groupby("age_group", observed=True)["survived"].agg(["count", "mean"])
    by_family = df.groupby("family_size")["survived"].agg(["count", "mean"])
    by_alone = df.groupby("travels_alone")["survived"].agg(["count", "mean"])
    return {"by_age": by_age.round(2), "by_family": by_family.round(2),
            "by_alone": by_alone.round(2)}

def analysis(df):
    return summarize(add_features(clean(df)))

results = analysis(titanic)
display(results["by_age"])
display(results["by_family"])
display(results["by_alone"])

## Exercise 6 — Statistical traps

Four mini-cases where the data seem to say one thing and actually say another. The files are in the `Datasets/` folder.

| Part | Dataset | Trap |
|---|---|---|
| 6a | `icecream_sharks.csv` (synthetic) | correlation ≠ causation: the confounder |
| 6b | `berkeley_admissions.csv` (real, 1973) | Simpson's paradox |
| 6c | `galton_heights.csv` (real, 1886) + simulation | regression to the mean |
| 6d | `anscombe.csv` | same statistics, very different data |

In [ ]:
import numpy as np

### 6a — Do ice creams cause shark attacks?

Weekly data 2020–2024 from a seaside town (**synthetic**: we know how they were generated).

```
Do ice creams cause attacks?
├── Raw correlation           → ice_creams_sold vs shark_attacks
├── Look for a third wheel    → correlation matrix of all columns
├── Stratify                  → correlation within temperature bands (pd.qcut)
└── Adjust                    → correlation between residuals after "removing" temperature
```

**To do:** `corr_by_stratum(df, x, y, z, n_bins)` and `partial_correlation(df, x, y, z)`
(hint: the residuals of a line `np.polyfit(z, x, 1)` are the part of `x` that `z` does not explain).

In [ ]:
sharks = pd.read_csv("../Datasets/icecream_sharks.csv")

def corr_by_stratum(df, x, y, z, n_bins=3):
    bins = pd.qcut(df[z], n_bins)
    return df.groupby(bins, observed=True).apply(lambda g: round(g[x].corr(g[y]), 2))

def residuals(df, y, z):
    slope, intercept = np.polyfit(df[z], df[y], 1)
    return df[y] - (intercept + slope * df[z])

def partial_correlation(df, x, y, z):
    return round(residuals(df, x, z).corr(residuals(df, y, z)), 2)

display(sharks.drop(columns="week").corr().round(2))
print("Raw correlation ice creams-attacks:", round(sharks.ice_creams_sold.corr(sharks.shark_attacks), 2))
print(corr_by_stratum(sharks, "ice_creams_sold", "shark_attacks", "mean_temperature_c", 5))
print("Partial correlation | temperature:",
      partial_correlation(sharks, "ice_creams_sold", "shark_attacks", "mean_temperature_c"))

#### Statistical explanation

**The model that generated the data** (we know it because it is synthetic):

```
temperature ──► ice_creams_sold
     │
     └──► swimmers ──► shark_attacks
```

- `ice_creams = 400 + 95·T + noise`
- `swimmers = 300·(T − 12) · noise`
- `attacks ~ Poisson(swimmers / 2200)`

Ice creams **do not appear** in the equation for attacks, yet the two variables have a correlation of about 0.55.
Temperature is a **confounder**: a common cause of both. In the language of causal graphs (DAGs) it opens a
*backdoor path* ice creams ← temperature → swimmers → attacks, which creates an association without any direct causal link.

**Two ways to "close" the path:**
1. **Stratification:** within bands of similar temperature the correlation drops to ±0.1–0.2. It does not get exactly to 0
   because within each band temperature still varies a bit (*residual confounding*): the narrower the bands, the better it works.
2. **Partial correlation:** correlate what is left of ice creams and attacks after removing the part linearly explained by T.
   It is equivalent to the formula
   $$r_{xy\cdot z} = \frac{r_{xy} - r_{xz}\,r_{yz}}{\sqrt{(1-r_{xz}^2)(1-r_{yz}^2)}}$$
   and here it is about 0.

**Careful:** "controlling for everything" is not the answer. Controlling for a *mediator* (swimmers, if we wanted the effect
of temperature on attacks) or for a *collider* (a variable caused by both) introduces bias.
Which variables to control for is decided by the causal model, not by the data alone.

#### References
- J. Pearl, M. Glymour, N. P. Jewell, *Causal Inference in Statistics: A Primer*, Wiley, 2016 (ch. 1–3).
- J. Pearl, D. Mackenzie, *The Book of Why*, Basic Books, 2018 (popular science).
- M. A. Hernán, J. M. Robins, *Causal Inference: What If*, CRC Press, 2020 (freely available on Hernán's website).

### 6b — Did Berkeley discriminate against women?

Applications to the 6 largest departments of UC Berkeley, fall 1973 (4526 applicants).
Careful: the file is in **aggregated format**, each row is a count (`n`), not a person.

```
Is there discrimination in admissions?
├── Admission rate by gender, overall
├── Admission rate by gender, within each department
├── Where do men and women apply?          → % of applications per department
└── How selective is each department?      → overall admission rate per department
```

**To do:** `admission_rate(df, groups)` that uses the `n` column as a weight
(why would `groupby(...).mean()` give a wrong result here?).

In [ ]:
berkeley = pd.read_csv("../Datasets/berkeley_admissions.csv")

def admission_rate(df, groups):
    total = df.groupby(groups)["n"].sum()
    admitted = df[df.outcome == "admitted"].groupby(groups)["n"].sum()
    return (admitted / total * 100).round(1)

def application_share(df):
    apps = df.groupby(["gender", "department"])["n"].sum().unstack()
    return (apps.div(apps.sum(axis=1), axis=0) * 100).round(1)

print("Overall:"); display(admission_rate(berkeley, ["gender"]))
print("By department:"); display(admission_rate(berkeley, ["department", "gender"]).unstack())
print("% of applications per department:"); display(application_share(berkeley))
print("Selectivity:"); display(admission_rate(berkeley, ["department"]))

#### Statistical explanation

Overall, 45% of men and 30% of women are admitted, but in 4 departments out of 6 women have an equal or higher rate
(in department A: 82% vs 62%). This is not a contradiction, it is the arithmetic of **weights**:

$$P(\text{admitted}\mid g) = \sum_d P(\text{admitted}\mid g, d)\; P(d\mid g)$$

The overall rate of each gender is a **weighted average** of the department rates, but with different weights:
93% of women apply to departments C–F, which admit 6–35% of applicants, while half of the men
apply to A and B, which admit about 64%. The department acts as a confounder between gender and outcome.

This is **Simpson's paradox**: an association present in every subgroup weakens, disappears or reverses when the groups
are aggregated. It happens when the stratification variable is associated with both the group and the outcome.

**Which analysis is the "right" one?** It depends on the question. If we ask "do the department committees discriminate?", we stratify
by department (and the data's answer is: no, if anything there is a slight advantage for women). If we ask "why do fewer women get in?",
the answer lies upstream: in the choice of programme, which has its own social causes. Bickel and colleagues concluded that the imbalance
did not come from the committees but from an earlier selection (career guidance, educational paths).
For a single summary measure adjusted for department, use the **Mantel–Haenszel odds ratio**.

#### References
- P. J. Bickel, E. A. Hammel, J. W. O'Connell, "Sex bias in graduate admissions: Data from Berkeley", *Science* 187(4175):398–404, 1975.
- E. H. Simpson, "The interpretation of interaction in contingency tables", *Journal of the Royal Statistical Society B* 13(2):238–241, 1951.
- Pearl & Mackenzie, *The Book of Why*, ch. 6 (causal discussion of the paradox).

### 6c — The cover jinx (regression to the mean)

**Part 1 — Galton.** Mid-parent height and height of the adult children (928 pairs, converted to cm).

```
Are children of tall parents just as tall?
├── Scatter plot parents vs children + regression line
├── Slope of the line children ~ parents         (is it 1?)
├── Take the tallest 10% of parents              → how far above the mean are they? and their children?
└── Slope of the line parents ~ children         (surprise!)
```

**Part 2 — "Sports Illustrated" simulation.** 500 athletes; the performance in a season is `skill + luck`,
both drawn from a normal N(0, 1). Put the 10 best of season 1 on the cover: how do they do in season 2?

```
Does the cover bring bad luck?
├── generate_skill(n)            → fixed skill of each athlete
├── simulate_season(skill)       → skill + luck (new every season)
├── pick_cover(perf, k)          → indices of the k best
└── compare                      → season 1 vs season 2 mean of the cover athletes
```

In [ ]:
galton = pd.read_csv("../Datasets/galton_heights.csv")
x, y = galton["midparent_height_cm"], galton["child_height_cm"]

# Part 1
slope, intercept = np.polyfit(x, y, 1)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
jitter = np.random.default_rng(1).normal(0, 0.4, (2, len(x)))  # the data are rounded into classes
ax.scatter(x + jitter[0], y + jitter[1], alpha=0.25, s=12)
xs = np.linspace(x.min(), x.max(), 10)
ax.plot(xs, intercept + slope * xs, color="C1", label=f"regression (slope {slope:.2f})")
ax.plot(xs, xs, "--", color="grey", label="child = parents (slope 1)")
ax.set_xlabel("mid-parent height (cm)"); ax.set_ylabel("child height (cm)"); ax.legend()
plt.show()

tall = galton[x >= x.quantile(0.9)]
print(f"Top 10% parents: +{tall.iloc[:, 0].mean() - x.mean():.1f} cm above the mean")
print(f"Their children:  +{tall.iloc[:, 1].mean() - y.mean():.1f} cm above the mean")
print(f"Slope children ~ parents: {slope:.2f}")
print(f"Slope parents ~ children: {np.polyfit(y, x, 1)[0]:.2f}  <- it regresses 'backwards' too")

# Part 2
rng = np.random.default_rng(0)

def generate_skill(n):
    return rng.normal(0, 1, n)

def simulate_season(skill):
    return skill + rng.normal(0, 1, len(skill))

def pick_cover(performances, k=10):
    return np.argsort(performances)[-k:]

skill = generate_skill(500)
s1, s2 = simulate_season(skill), simulate_season(skill)
cover = pick_cover(s1)
print(f"\nCover athletes - season 1: {s1[cover].mean():.2f}  season 2: {s2[cover].mean():.2f}")
print(f"True skill of the cover athletes: {skill[cover].mean():.2f}")
print(f"{np.mean(s2[cover] < s1[cover]) * 100:.0f}% of the cover athletes get worse, with no jinx at all.")

#### Statistical explanation

**The model.** Every measurement is *stable component + random component*: $Y_t = S + \varepsilon_t$, with $S \sim N(\mu, \sigma_S^2)$
and $\varepsilon_t \sim N(0, \sigma_\varepsilon^2)$ independent across seasons. The expected value of the next season,
given how this one went, is

$$E[Y_2 \mid Y_1 = y] = \mu + \rho\,(y - \mu), \qquad \rho = \frac{\sigma_S^2}{\sigma_S^2 + \sigma_\varepsilon^2}$$

where $\rho$ (*reliability*, equal to the correlation between seasons) is < 1 as soon as there is some chance involved. In the simulation
$\sigma_S = \sigma_\varepsilon = 1$, so ρ = 0.5: someone who was 3 standard deviations above the mean is expected to come back to about 1.5.
Whoever ends up at the top of a ranking is, on average, both skilled **and lucky**, and luck does not repeat itself.

**Galton.** Slope 0.65 (< 1): children of very tall parents are tall, but closer to the mean. Galton called it
*regression towards mediocrity*, and this is where the name "regression" for the least-squares line comes from.
The proof that it is a statistical and not a biological phenomenon: the line *parents ~ children* also has slope < 1 (≈ 0.33).
The parents of very tall children are less extreme than the children. Note: in Galton's data female heights are multiplied
by 1.08 to make them comparable, and the values are rounded into classes (hence the *jitter* in the plot).

**The Sports Illustrated "jinx".** US sports legend: whoever appears on the cover of *Sports Illustrated* soon afterwards
has a slump or an injury. The first case usually cited is Eddie Mathews, on the cover of the first issue (16 August 1954):
a few days later his team's 9-game winning streak ended and he injured his hand.
In 2002 the magazine dedicated a cover to it, with a black cat ("The Cover No One Would Pose For"), and an investigation:
out of 2456 covers, in 913 cases (37.2%) a documentable misfortune followed.
But athletes end up on the cover **after** an exceptional performance, i.e. selected on the extreme of $Y_1$: the following drop is
exactly what regression to the mean predicts. The cover is a marker of the peak, not its cause.

**Where it shows up in real life:**
- *Medicine:* patients are enrolled because their blood pressure is very high; at the next visit it is lower even without any drug.
  This is why control groups are needed.
- *Kahneman and the flight instructors:* after praise for an excellent landing the next one is worse, after a scolding
  for a terrible landing the next one is better. The instructors concluded that scolding works and praise does not.
- *Public policy:* speed cameras installed at the junctions with the most accidents the year before; the year after, accidents go down anyway.

#### References
- F. Galton, "Regression towards mediocrity in hereditary stature", *Journal of the Anthropological Institute* 15:246–263, 1886.
- S. M. Stigler, "Regression towards the mean, historically considered", *Statistical Methods in Medical Research* 6:103–114, 1997.
- A. G. Barnett, J. C. van der Pols, A. J. Dobson, "Regression to the mean: what it is and how to deal with it",
  *International Journal of Epidemiology* 34(1):215–220, 2005.
- D. Kahneman, *Thinking, Fast and Slow*, 2011, ch. 17.
- A. Wolff, "That Old Black Magic", *Sports Illustrated*, 21 January 2002, pp. 50–61.

### 6d — Can you trust summary statistics? (Anscombe)

Four datasets (`I`, `II`, `III`, `IV`) of 11 points each.

```
Are the four datasets the same?
├── For each dataset: mean and variance of x and y, correlation, regression line
└── 2×2 grid of scatter plots
```

**To do:** `summary_stats(df)` to apply with `groupby("dataset")`, then the grid of plots.
Before plotting, make a bet: do the datasets look alike?

In [ ]:
anscombe = pd.read_csv("../Datasets/anscombe.csv")

def summary_stats(g):
    slope, intercept = np.polyfit(g.x, g.y, 1)
    return pd.Series({"mean_x": g.x.mean(), "var_x": g.x.var(), "mean_y": g.y.mean(),
                      "var_y": g.y.var(), "corr": g.x.corr(g.y),
                      "slope": slope, "intercept": intercept}).round(2)

display(anscombe.groupby("dataset")[["x", "y"]].apply(summary_stats))

fig, axes = plt.subplots(2, 2, figsize=(8, 6), sharex=True, sharey=True)
for ax, (name, g) in zip(axes.flat, anscombe.groupby("dataset")):
    p, q = np.polyfit(g.x, g.y, 1)
    ax.scatter(g.x, g.y); ax.plot([3, 20], [q + 3 * p, q + 20 * p], color="C1")
    ax.set_title(f"Dataset {name}")
plt.tight_layout(); plt.show()

#### Statistical explanation

The four datasets have (up to the second decimal place) the same means, variances, correlation (≈ 0.82) and line
$y = 3 + 0.5x$, but they tell different stories:
- **I:** linear relationship with noise, the only case in which the line is a good model;
- **II:** curved (parabolic) relationship: the linear model is wrong;
- **III:** perfect linear relationship, except for one **outlier** that shifts the line;
- **IV:** x is constant except for one point: the "correlation" is produced entirely by a single high-*leverage* observation.

Summary statistics compress the data and lose information; two different distributions can have the same moments.
Pearson's correlation only measures the **linear** component and is sensitive to extreme values
(more robust alternatives: Spearman, Kendall). The moral for EDA: **look first, then summarize.**

Modern version: the *Datasaurus Dozen*, 13 datasets with the same statistics, one of which draws a dinosaur.

#### References
- F. J. Anscombe, "Graphs in statistical analysis", *The American Statistician* 27(1):17–21, 1973.
- J. Matejka, G. Fitzmaurice, "Same Stats, Different Graphs: Generating Datasets with Varied Appearance and Identical
  Statistics through Simulated Annealing", *Proc. CHI 2017*, ACM.